# InsightForge AI — 00 Agent Setup & State Definition
Defines shared state (`InsightForgeState`), structured logging, and Gemini LLM initialization.


In [ ]:
%pip install pandas plotly fpdf2 langgraph langchain google-generativeai
dbutils.library.restartPython()


In [ ]:
"""
InsightForge AI — Logging Setup
================================
Configures structured logging for the full pipeline.
Every agent logs its start, completion, and any errors.

Log levels used:
- INFO    : normal operation messages
- WARNING : non-critical issues (rate limits, missing cols)
- ERROR   : agent failures that affect output quality
"""

import logging
import sys
from datetime import datetime

# ── Configure logger ──────────────────────────────────────────
def setup_logger(name: str = "InsightForge") -> logging.Logger:
    """
    Creates and configures a logger for the pipeline.
    Logs to console with timestamp, level, and message.
    Returns the same logger if already configured so
    calling setup_logger() multiple times is safe.

    Parameters
    ----------
    name : str — logger name, default InsightForge

    Returns
    -------
    logging.Logger : configured logger instance
    """
    logger = logging.getLogger(name)

    # Avoid adding duplicate handlers on re-run
    if logger.handlers:
        return logger

    logger.setLevel(logging.DEBUG)

    # ── Console handler ───────────────────────────────────────
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)

    formatter = logging.Formatter(
        fmt   = "%(asctime)s  [%(levelname)-8s]  %(message)s",
        datefmt = "%H:%M:%S"
    )
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    return logger


# ── Initialise logger ─────────────────────────────────────────
logger = setup_logger("InsightForge")

logger.info("InsightForge AI logging initialised")
logger.info(f"Dataset  : {DATASET_PATH}")
logger.info(f"Model    : {GEMINI_MODEL}")
logger.info(f"Log level: INFO")

print()
print("✅ Logger configured")
print("   Format: HH:MM:SS  [LEVEL]  message")


In [ ]:
class InsightForgeState(TypedDict):
    """
    Shared state passed through all agents in the pipeline.
    Each agent reads what it needs and writes its output.
    No agent modifies another agent's output fields.

    Fields
    ------
    dataset_path     : path to the input CSV file
    gemini_key       : Gemini API key passed at runtime
    raw_df           : original DataFrame as uploaded
    cleaned_df       : DataFrame after cleaning agent runs
    schema_info      : column metadata detected by schema agent
    cleaning_report  : summary of all cleaning actions taken
    eda_results      : statistical analysis from EDA agent
    charts           : list of Plotly figure dicts from viz agent
    insights         : AI generated business insights text
    pdf_path         : path to the generated PDF report
    pipeline_log     : timestamped log of each agent execution
    errors           : list of error messages from any agent
    """
    dataset_path    : str
    gemini_key      : str
    raw_df          : Any
    cleaned_df      : Any
    schema_info     : dict
    cleaning_report : dict
    eda_results     : dict
    charts          : list
    insights        : str
    pdf_path        : str
    pipeline_log    : list
    errors          : list

print("✅ InsightForgeState defined")
print()
print("  State fields:")
fields = [
    ("dataset_path",     "input — path to CSV"),
    ("gemini_key",       "input — API key"),
    ("raw_df",           "Schema Agent reads this"),
    ("cleaned_df",       "Cleaning Agent writes this"),
    ("schema_info",      "Schema Agent writes this"),
    ("cleaning_report",  "Cleaning Agent writes this"),
    ("eda_results",      "EDA Agent writes this"),
    ("charts",           "Visualization Agent writes this"),
    ("insights",         "Insight Agent writes this"),
    ("pdf_path",         "Report Agent writes this"),
    ("pipeline_log",     "every agent appends to this"),
    ("errors",           "every agent appends on failure"),
]
for field, desc in fields:
    print(f"    {field:20} — {desc}")


In [ ]:
def log_event(state: InsightForgeState, agent: str, message: str) -> list:
    """
    Appends a timestamped log entry to the pipeline log.
    Called by every agent on start and completion.

    Parameters
    ----------
    state   : current pipeline state
    agent   : name of the calling agent
    message : what happened

    Returns
    -------
    list : updated pipeline log
    """
    timestamp = datetime.now().strftime("%H:%M:%S")
    entry     = f"[{timestamp}] {agent}: {message}"
    print(f"   {entry}")
    return state["pipeline_log"] + [entry]


def get_gemini_model() -> genai.GenerativeModel:
    """
    Returns a configured Gemini model instance.
    Re-reads the API key from widget each time to handle
    session restarts without needing to re-run setup cells.
    """
    key = dbutils.widgets.get("gemini_key")
    genai.configure(api_key=key)
    return genai.GenerativeModel(GEMINI_MODEL)


def safe_call_gemini(prompt: str, agent_name: str) -> str:
    """
    Wraps a Gemini API call with error handling.
    Cleans the response text to remove characters that
    fpdf2 cannot render with standard Helvetica font.

    Parameters
    ----------
    prompt     : the full prompt string to send
    agent_name : name of the calling agent for logging

    Returns
    -------
    str : cleaned response text or error message
    """
    try:
        m        = get_gemini_model()
        response = m.generate_content(prompt)
        text     = response.text

        # Remove characters unsupported by Helvetica in fpdf2
        replacements = {
            "\u2014": "-",    # em dash
            "\u2013": "-",    # en dash
            "\u2012": "-",    # figure dash
            "\u2011": "-",    # non-breaking hyphen
            "\u2010": "-",    # hyphen
            "\u2022": "-",    # bullet
            "\u2023": "-",    # triangle bullet
            "\u2043": "-",    # hyphen bullet
            "\u2018": "'",    # left single quote
            "\u2019": "'",    # right single quote
            "\u201a": "'",    # single low quote
            "\u201c": '"',    # left double quote
            "\u201d": '"',    # right double quote
            "\u201e": '"',    # double low quote
            "\u2026": "...",  # ellipsis
            "\u00a0": " ",    # non-breaking space
            "\u00b7": "-",    # middle dot
            "\u2015": "-",    # horizontal bar
        }
        for char, replacement in replacements.items():
            text = text.replace(char, replacement)

        # Final safety pass — replace remaining non-latin-1 chars
        text = text.encode("latin-1", errors="replace").decode("latin-1")
        return text

    except Exception as e:
        logger.warning(
            f"Gemini call failed in {agent_name}: {str(e)[:100]}"
        )
        print(f"   ⚠️  Gemini call failed in {agent_name}: {e}")
        return f"[Gemini error in {agent_name}: {str(e)}]"


print("✅ Utility functions defined")
print("   log_event()        — timestamped pipeline logging")
print("   get_gemini_model() — safe model initialisation")
print("   safe_call_gemini() — error handled API call with font cleaning")
